# Raw Ring IMU viewer

Select one recording with `USER`, `ACTION`, and `DATASET_ID`, then load only its primary `ring_0` stream. The notebook plots the raw timestamp column, the three accelerometer channels, and the three gyroscope channels.

The raw signal units are not established by this repository. The loader also exposes an inferred relative-time column, but its microsecond interpretation is explicitly unconfirmed.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from writingring.discovery import discover_recordings
from writingring.ring_loader import RELATIVE_TIME_COLUMN, load_ring
from writingring.selection import select_recording

DATA_ROOT = PROJECT_ROOT / "data"

# Change these three values to inspect another recording.
USER = "user_0"
ACTION = "0"
DATASET_ID = 0

# Use "sample_index" by default because raw timestamps may contain duplicates
# and their upstream unit is not confirmed. Set to "inferred_time" when
# an inferred relative-time axis is useful for visual inspection.
SIGNAL_X_AXIS = "sample_index"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "diagnostics" / "raw_ring_imu"
SAVE_FIGURES = False
FIGURE_DPI = 150

if SIGNAL_X_AXIS not in {"sample_index", "inferred_time"}:
    raise ValueError("SIGNAL_X_AXIS must be 'sample_index' or 'inferred_time'")

recordings = discover_recordings(DATA_ROOT)
recording = select_recording(
    recordings, user=USER, action=ACTION, dataset_id=DATASET_ID
)
ring_data = load_ring(recording)
data = ring_data.dataframe

assert ring_data.source_path == recording.ring_0_path
required_columns = {"acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "timestamp"}
missing = required_columns.difference(data.columns)
if missing:
    raise ValueError(f"Loaded Ring data is missing columns: {sorted(missing)}")

print(f"Selected: {recording.user} / action {recording.action} / dataset {recording.dataset_id}")
print(f"Primary payload: {ring_data.source_path}")
print(f"Shape: {data.shape}")
print(f"Raw timestamp range: {data['timestamp'].iloc[0]:.0f} .. {data['timestamp'].iloc[-1]:.0f}")
print(f"Duplicate timestamp steps: {ring_data.validation.duplicate_timestamp_steps}")
print(f"Backward timestamp steps: {ring_data.validation.backward_timestamp_steps}")
print(f"Ring 1 loaded: no; only ring_0 is used")

In [ ]:
# Plot the raw timestamp values in source row order.
sample_index = np.asarray(data.index, dtype=np.int64)
raw_timestamp = data["timestamp"].to_numpy(dtype=np.float64, copy=True)

fig, ax = plt.subplots(figsize=(15, 4.5), layout="constrained")
ax.plot(sample_index, raw_timestamp, color="tab:purple", linewidth=0.8)
ax.set_title(
    f"Raw Ring timestamp | {recording.user} / action {recording.action} / dataset {recording.dataset_id}"
)
ax.set_xlabel("Sample index")
ax.set_ylabel("Raw timestamp value (stored units)")
ax.grid(True, alpha=0.3)
plt.show()

if SAVE_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"{USER}_action_{ACTION}_{DATASET_ID}_timestamp.png", dpi=FIGURE_DPI)


In [ ]:
# Plot accelerometer and gyroscope channels in separate panels.
if SIGNAL_X_AXIS == "sample_index":
    x_values = sample_index
    x_label = "Sample index"
else:
    if RELATIVE_TIME_COLUMN not in data.columns:
        raise ValueError(
            f"{RELATIVE_TIME_COLUMN!r} is unavailable; use SIGNAL_X_AXIS='sample_index'"
        )
    x_values = data[RELATIVE_TIME_COLUMN].to_numpy(dtype=np.float64, copy=True)
    x_label = (
        "Inferred relative time (s; microsecond interpretation unconfirmed)"
    )

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(15, 8), layout="constrained")
for column in ("acc_x", "acc_y", "acc_z"):
    axes[0].plot(x_values, data[column].to_numpy(dtype=np.float64, copy=True), linewidth=0.8, label=column)
for column in ("gyr_x", "gyr_y", "gyr_z"):
    axes[1].plot(x_values, data[column].to_numpy(dtype=np.float64, copy=True), linewidth=0.8, label=column)

axes[0].set_ylabel("Acceleration raw value")
axes[1].set_ylabel("Gyroscope raw value")
axes[1].set_xlabel(x_label)
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.suptitle(
    f"Raw Ring IMU | {recording.user} / action {recording.action} / dataset {recording.dataset_id}"
)
plt.show()

if SAVE_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"{USER}_action_{ACTION}_{DATASET_ID}_accel_gyro.png", dpi=FIGURE_DPI)


In [ ]:
# Compact validation/preview. This does not sort, repair, or rewrite the data.
display(data.loc[:, ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "timestamp"]].head())
print("Warnings:")
for warning in ring_data.warnings:
    print(f"- {warning}")
